In [ ]:
from typing import TypedDict
from langchain_core.messages import SystemMessage
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode

# -------------------------
# STATE
# -------------------------
class AgentState(MessagesState):
    next_agent: str  # supervisor decides this


# -------------------------
# TOOL
# -------------------------
@tool
def search_web(query: str) -> str:
    """Search the web for information"""
    search = TavilySearchResults(max_results=3)
    results = search.invoke(query)
    return str(results)


llm = ChatOpenAI(model="gpt-3.5-turbo")


# -------------------------
# SUPERVISOR AGENT
# -------------------------
def supervisor_agent(state: AgentState):
    """Supervisor decides which agent should run next."""

    messages = state["messages"]

    system_msg = SystemMessage(
        content=(
            "You are a supervisor AI. Decide which agent should act next.\n"
            "Options:\n"
            "- researcher → if more information is needed\n"
            "- writer → if a summary or refinement is needed\n"
            "- end → if the task is complete\n"
            "Respond ONLY with one word: researcher, writer, or end."
        )
    )

    response = llm.invoke([system_msg] + messages)
    decision = response.content.strip().lower()

    if decision not in ["researcher", "writer", "end"]:
        decision = "writer"

    return {
        "messages": [response],
        "next_agent": decision
    }


# -------------------------
# RESEARCHER AGENT
# -------------------------
def researcher_agent(state: AgentState):
    """Researcher agent that searches for information"""

    messages = state["messages"]

    system_msg = SystemMessage(
        content="You are a research assistant. Use the search_web tool to find relevant information."
    )

    researcher_llm = llm.bind_tools([search_web])
    response = researcher_llm.invoke([system_msg] + messages)

    # If tool call exists → route to tools
    if hasattr(response, "tool_calls") and response.tool_calls:
        return {
            "messages": [response],
            "next_agent": "tools"
        }

    # Otherwise → return to supervisor
    return {
        "messages": [response],
        "next_agent": "supervisor"
    }


# -------------------------
# WRITER AGENT (NOW USES TOOLS)
# -------------------------
def writer_agent(state: AgentState):
    """Writer agent that creates summary and may use tools."""

    messages = state["messages"]

    system_message = SystemMessage(
        content="You are a technical writer. You may use tools if needed to improve the summary."
    )

    writer_llm = llm.bind_tools([search_web])
    response = writer_llm.invoke([system_message] + messages)

    # If writer triggers a tool call → go to tools
    if hasattr(response, "tool_calls") and response.tool_calls:
        return {
            "messages": [response],
            "next_agent": "tools"
        }

    # Otherwise → return to supervisor
    return {
        "messages": [response],
        "next_agent": "supervisor"
    }


# -------------------------
# BUILD GRAPH
# -------------------------
workflow = StateGraph(AgentState)

workflow.add_node("supervisor", supervisor_agent)
workflow.add_node("researcher", researcher_agent)
workflow.add_node("writer", writer_agent)
workflow.add_node("tools", ToolNode([search_web]))

# FLOW
workflow.add_edge(START, "supervisor")

workflow.add_conditional_edges(
    "supervisor",
    lambda state: state["next_agent"],
    {
        "researcher": "researcher",
        "writer": "writer",
        "end": END
    }
)

workflow.add_conditional_edges(
    "researcher",
    lambda state: state["next_agent"],
    {
        "tools": "tools",
        "supervisor": "supervisor"
    }
)

workflow.add_conditional_edges(
    "writer",
    lambda state: state["next_agent"],
    {
        "tools": "tools",
        "supervisor": "supervisor"
    }
)

workflow.add_edge("tools", "supervisor")

# COMPILE
final_workflow = workflow.compile()

# RUN
response = final_workflow.invoke({
    "messages": "Research about the usecases of agentic AI in the IT industry"
})

for msg in response["messages"]:
    msg.pretty_print()


In [ ]:
from IPython.display import Image, display

display(Image(final_workflow.get_graph().draw_mermaid_png())) 